In [1]:
import sys, os
import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings("ignore") 

src_path = os.path.abspath("../src")
if src_path not in sys.path:
    sys.path.append(src_path)
%load_ext autoreload
%autoreload 2

print("Setup complete. src in path:", src_path in sys.path)

Setup complete. src in path: True


In [9]:
SAMPLE_DIR = os.path.abspath("../data/raw_sample")

DATASET = "diabetes_sliced.csv"
TARGET_COL = "diabetes"

MODELS_DIR = os.path.abspath("../models")
RANDOM_STATE = 42

print(f"Dataset: {DATASET} | Target: {TARGET_COL}")


Dataset: diabetes_sliced.csv | Target: diabetes


In [10]:
from data_cleaning import clean_dataset
from feature_engineering import run_pipeline

df_raw = pd.read_csv(os.path.join(SAMPLE_DIR, DATASET))
df_clean, _ = clean_dataset(df_raw)


X_train, X_test, y_train, y_test, artifacts, reports = run_pipeline(
    df_clean, target_col=TARGET_COL,
    processed_dir="../data/processed",
    models_dir=MODELS_DIR,
    random_state=RANDOM_STATE,
)

feature_columns = artifacts["feature_columns"]
print("\nReady:")
print("  X_train:", X_train.shape, "| X_test:", X_test.shape)
print("  features:", len(feature_columns))
print("  class distribution (train):", y_train.value_counts().to_dict())

2026-07-07 15:04:36,495 | data_cleaning | INFO | Cleaning started. Input shape: (20000, 9)
2026-07-07 15:04:36,676 | data_cleaning | INFO | Removed 196 duplicate rows
2026-07-07 15:04:36,818 | data_cleaning | INFO | Skipped 'hypertension' for outliers (2 unique values)
2026-07-07 15:04:36,824 | data_cleaning | INFO | Skipped 'heart_disease' for outliers (2 unique values)
2026-07-07 15:04:36,845 | data_cleaning | INFO | Capped 1492 outliers in 'bmi'
2026-07-07 15:04:36,856 | data_cleaning | INFO | Capped 244 outliers in 'HbA1c_level'
2026-07-07 15:04:36,867 | data_cleaning | INFO | Capped 406 outliers in 'blood_glucose_level'
2026-07-07 15:04:36,870 | data_cleaning | INFO | Skipped 'diabetes' for outliers (2 unique values)
2026-07-07 15:04:36,872 | data_cleaning | INFO | Cleaning finished. Output shape: (19804, 9)
2026-07-07 15:04:36,876 | feature_engineering | INFO | Pipeline started. Input shape: (19804, 9), target: diabetes
2026-07-07 15:04:37,002 | feature_engineering | INFO | Featu


Ready:
  X_train: (15843, 15) | X_test: (3961, 15)
  features: 15
  class distribution (train): {0: 14483, 1: 1360}


In [15]:
IMBALANCE_THRESHOLD = 0.40

def detect_imbalance(y, threshold=IMBALANCE_THRESHOLD):
     counts = y.value_counts()
     props = y.value_counts(normalize=True)
     minority_share = props.min()
     is_imbalanced = minority_share < threshold
     return {
        "class_counts": counts.to_dict(),
        "class_proportions": props.round(3).to_dict(),
        "n_classes": len(counts),
        "minority_share": round(minority_share, 3),
        "is_imbalanced": is_imbalanced,
        "class_weight": "balanced" if is_imbalanced else None,
     }

imbalance_report = detect_imbalance(y_train)

print("Class counts   :", imbalance_report["class_counts"])
print("Class proportions:", imbalance_report["class_proportions"])
print("Minority share   :", imbalance_report["minority_share"])
print(f"Imbalanced (<{IMBALANCE_THRESHOLD})?:", imbalance_report["is_imbalanced"])
print("class_weight  :", imbalance_report["class_weight"])






Class counts   : {0: 14483, 1: 1360}
Class proportions: {0: 0.914, 1: 0.086}
Minority share   : 0.086
Imbalanced (<0.4)?: True
class_weight  : balanced


In [18]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

def build_models(class_weight=None, random_state=RANDOM_STATE):
    return{
        "LogisticRegression": LogisticRegression(
            class_weight=class_weight, random_state=random_state, max_iter=1000
        ),
        "DecisionTree": DecisionTreeClassifier(
            class_weight=class_weight, random_state=random_state
        ),
        "RandomForest": RandomForestClassifier(
            class_weight=class_weight, random_state=random_state
        ),
        "GradientBoosting": GradientBoostingClassifier(
            random_state=random_state
        ),
    }

models = build_models(class_weight=imbalance_report["class_weight"])
print("Built models (class_weight =", imbalance_report["class_weight"], "):")
for name in models:
    print(" -", name)


Built models (class_weight = balanced ):
 - LogisticRegression
 - DecisionTree
 - RandomForest
 - GradientBoosting


In [19]:
from sklearn.utils.class_weight import compute_sample_weight

gb_sample_weight = None
if imbalance_report["class_weight"] == "balanced":
    gb_sample_weight = compute_sample_weight(class_weight="balanced", y=y_train)

trained_models = {}
training_report = {}

for name, model in models.items():
    try:
        if name == "GradientBoosting" and gb_sample_weight is not None:
            model.fit(X_train, y_train, sample_weight=gb_sample_weight)
            weight_status = "sample_weight=balanced"
        else:
            model.fit(X_train, y_train)
            weight_status = f"class_weight={imbalance_report['class_weight']}" \
                            if name != "GradientBoosting" else "none"
        trained_models[name] = model
        training_report[name] = {"trained": True, "weighting": weight_status}
    except Exception as e:
        training_report[name] = {"trained": False, "error": str(e)}

print("Training complete:")
for name, info in training_report.items():
    print(f"  {name:20s} | {info}")

Training complete:
  LogisticRegression   | {'trained': True, 'weighting': 'class_weight=balanced'}
  DecisionTree         | {'trained': True, 'weighting': 'class_weight=balanced'}
  RandomForest         | {'trained': True, 'weighting': 'class_weight=balanced'}
  GradientBoosting     | {'trained': True, 'weighting': 'sample_weight=balanced'}


In [21]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix,
)

n_classes = y_train.nunique()
is_binary = (n_classes == 2)
avg = "binary" if is_binary else "macro"

eval_report = {}
rows = []

for name, model in trained_models.items():
    try:
        y_pred = model.predict(X_test)
        if is_binary:
            prec = precision_score(y_test, y_pred, pos_label=1, zero_division=0)
            rec  = recall_score(y_test, y_pred, pos_label=1, zero_division=0)
            f1   = f1_score(y_test, y_pred, pos_label=1, zero_division=0)
        else:
            prec = precision_score(y_test, y_pred, average=avg, zero_division=0)
            rec  = recall_score(y_test, y_pred, average=avg, zero_division=0)
            f1   = f1_score(y_test, y_pred, average=avg, zero_division=0)
        acc = accuracy_score(y_test, y_pred)

        auc = np.nan
        if hasattr(model, "predict_proba"):
            proba = model.predict_proba(X_test)
            auc = (roc_auc_score(y_test, proba[:, 1]) if is_binary
                   else roc_auc_score(y_test, proba, multi_class="ovr", average="macro"))

        cm = confusion_matrix(y_test, y_pred)
        eval_report[name] = {"accuracy": acc, "precision": prec, "recall": rec,
                             "f1": f1, "roc_auc": auc, "confusion_matrix": cm}
        rows.append({"model": name, "accuracy": acc, "precision": prec,
                     "recall": rec, "f1": f1, "roc_auc": auc})
    except Exception as e:
        eval_report[name] = {"error": str(e)}

comparison_df = (pd.DataFrame(rows).sort_values("f1", ascending=False)
                 .reset_index(drop=True).round(3))

print(f"Problem type: {'binary' if is_binary else f'multiclass ({n_classes})'}")
print("\n=== Model Comparison (sorted by F1) ===")
print(comparison_df.to_string(index=False))

print("\n=== Confusion Matrices (rows=actual, cols=predicted) ===")

Problem type: binary

=== Model Comparison (sorted by F1) ===
             model  accuracy  precision  recall    f1  roc_auc
      RandomForest     0.963      0.800   0.765 0.782    0.968
      DecisionTree     0.959      0.771   0.744 0.757    0.862
  GradientBoosting     0.906      0.474   0.912 0.624    0.979
LogisticRegression     0.879      0.408   0.897 0.561    0.963

=== Confusion Matrices (rows=actual, cols=predicted) ===


In [22]:
best_name = comparison_df.iloc[0]["model"]
best_model = trained_models[best_name]
best_metrics = eval_report[best_name]

best_bundle = {
    "model": best_model,
    "model_name": best_name,
    "feature_columns": feature_columns,
    "target_col": TARGET_COL,
    "metrics": {
        "accuracy": round(best_metrics["accuracy"], 4),
        "precision": round(best_metrics["precision"], 4),
        "recall": round(best_metrics["recall"], 4),
        "f1": round(best_metrics["f1"], 4),
        "roc_auc": round(best_metrics["roc_auc"], 4) if not np.isnan(best_metrics["roc_auc"]) else None,
    },
}

os.makedirs(MODELS_DIR, exist_ok=True)
best_model_path = os.path.join(MODELS_DIR, "best_model.pkl")
joblib.dump(best_bundle, best_model_path)

print("Best model:", best_name)
print("Metrics   :", best_bundle["metrics"])
print("Saved     ->", best_model_path)

Best model: RandomForest
Metrics   : {'accuracy': 0.9634, 'precision': 0.8, 'recall': 0.7647, 'f1': 0.782, 'roc_auc': 0.9685}
Saved     -> d:\projects\Healthcare\models\best_model.pkl


In [23]:
os.makedirs("../reports", exist_ok=True)
report_path = "../reports/model_comparison.csv"
comparison_df.to_csv(report_path, index=False)

print("Model comparison report saved ->", report_path)
print("")
print(comparison_df.to_string(index=False))

Model comparison report saved -> ../reports/model_comparison.csv

             model  accuracy  precision  recall    f1  roc_auc
      RandomForest     0.963      0.800   0.765 0.782    0.968
      DecisionTree     0.959      0.771   0.744 0.757    0.862
  GradientBoosting     0.906      0.474   0.912 0.624    0.979
LogisticRegression     0.879      0.408   0.897 0.561    0.963
